In [0]:
spark.sql("USE CATALOG edtech_project")
spark.sql("USE SCHEMA edtech_bronze")

TASK 1.1

In [0]:
#read using autoloader
from pyspark.sql.functions import *

lms_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/edtech_project/edtech_bronze/edtech_project/lms_events/")
    .load("/Volumes/edtech_project/edtech_bronze/edtech_project/lms_events/")
)

In [0]:
#meta data
lms_df = lms_df.withColumn("_source_file", col("_metadata.file_path")) \
               .withColumn("_load_ts", current_timestamp()) \
               .withColumn("_schema_version", lit("v1"))


In [0]:
#data quality checks for not null and future date
lms_df = lms_df.filter(col("student_id").isNotNull()) \
               .filter(col("event_ts") <= current_timestamp())

In [0]:
#partition columns
lms_df = lms_df.withColumn("event_date", to_date("event_ts"))


In [0]:
%sql
-- displaying partitions
SHOW PARTITIONS lms_events_bronze;

In [0]:
#create table:
(lms_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/edtech_project/edtech_bronze/edtech_project/checkpoints/lms_events/")
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .table("lms_events_bronze")
)


In [0]:
%sql
-- display the table
SELECT * 
FROM lms_events_bronze
LIMIT 5;

In [0]:
%sql
SELECT count(*) 
FROM lms_events_bronze;